In [ ]:
%iam_role arn:aws:iam::770170581396:role/aws-glue-s3-permission
%region eu-north-1
%idle_timeout 15
%worker_type G.1X
%number_of_workers 2
%glue_version 4.0

# Bronze Layer — Development Notebook

**Dataset**: Amazon Fine Food Reviews — 10-column CSV uploaded to S3 raw folder

**Columns**: `Id`, `ProductId`, `UserId`, `ProfileName`, `HelpfulnessNumerator`, `HelpfulnessDenominator`, `Score`, `Time`, `Summary`, `Text`

**Steps**:
1. Read raw CSV from `s3://amazon-food-reviews-ml-model/raw/`
2. EDA — schema, nulls, Score distribution, text length, duplicates
3. Bronze transformation — rename, cast, derive label, add metadata

## 1. Read Raw CSV from S3

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType

# Full CSV already uploaded to S3 raw folder
RAW_PATH = "s3://amazon-food-reviews-ml-model/raw/amazon.csv"

# multiLine=True  -> handles review Text with embedded newlines
# escape='"'      -> correctly handles double-quoted fields
raw_df = (
    spark.read
    .option("header",      True)
    .option("inferSchema", True)
    .option("multiLine",   True)
    .option("escape",      '"')
    .option("quote",       '"')
    .csv(RAW_PATH)
)

raw_df.printSchema()
print(f"\nTotal rows: {raw_df.count()}")
raw_df.show(5, truncate=80)

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# ── 2a. Shape & descriptive statistics ──────────────────────────────────────
print("=" * 60)
print("SHAPE")
print("=" * 60)
print(f"  Rows    : {raw_df.count()}")
print(f"  Columns : {len(raw_df.columns)}")
print(f"  Names   : {raw_df.columns}")

print()
print("=" * 60)
print("DESCRIPTIVE STATS")
print("=" * 60)
raw_df.select(
    "HelpfulnessNumerator",
    "HelpfulnessDenominator",
    "Score"
).describe().show()

In [ ]:
# ── 2b. Null / missing value count per column ────────────────────────────────
print("=" * 60)
print("NULL COUNTS PER COLUMN")
print("=" * 60)
raw_df.select([
    F.count(F.when(
        F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""),
        c
    )).alias(c)
    for c in raw_df.columns
]).show()

In [ ]:
# ── 2c. Score distribution ──────────────────────────────────────────────────
print("=" * 60)
print("SCORE DISTRIBUTION  (1=worst  5=best)")
print("=" * 60)
raw_df.groupBy("Score").count().orderBy("Score").show()

# Preview the binary label we will derive
print("=" * 60)
print("BINARY LABEL PREVIEW  (Score >= 4 -> Positive=1, else 0)")
print("=" * 60)
raw_df.withColumn(
    "Positive",
    F.when(F.col("Score") >= 4, 1).otherwise(0)
).groupBy("Positive").count().orderBy("Positive").show()

In [ ]:
# ── 2d. Review text length analysis ────────────────────────────────────────
print("=" * 60)
print("REVIEW TEXT LENGTH STATS (characters)")
print("=" * 60)
raw_df.withColumn("text_len", F.length(F.col("Text"))).select(
    F.min("text_len").alias("min_chars"),
    F.round(F.avg("text_len"), 1).alias("avg_chars"),
    F.percentile_approx("text_len", 0.5).alias("median_chars"),
    F.max("text_len").alias("max_chars"),
).show()

# Suspiciously short reviews (< 10 chars)
short = raw_df.filter(F.length(F.col("Text")) < 10)
print(f"Reviews with < 10 characters: {short.count()}")
short.select("Id", "Score", "Text").show(10, truncate=False)

In [ ]:
# ── 2e. Duplicate detection ─────────────────────────────────────────────────
total    = raw_df.count()
distinct = raw_df.dropDuplicates(["Text"]).count()
print(f"Total rows            : {total}")
print(f"Distinct review texts : {distinct}")
print(f"Duplicate rows        : {total - distinct}")

## 3. Bronze Layer Transformation

Minimal, lossless cleaning applied to the raw data:

| Step | Detail |
|------|--------|
| **Select** | `Id`, `Score`, `Time`, `Summary`, `Text` only |
| **Rename** | `Text` → `review_text` · `Score` → `score` · `Summary` → `summary` |
| **Cast** | `score` → int · `time` → long |
| **Filter** | Drop null / empty `review_text`; keep `score` in 1–5 |
| **Derive label** | `positive = 1` if `score >= 4`, else `0` |
| **Metadata** | `_ingested_at` · `_source_file` · `_record_id` |

In [ ]:
from pyspark.sql.functions import (
    col, when, trim, length,
    current_timestamp, input_file_name, monotonically_increasing_id
)

bronze_df = (
    raw_df

    # ── 1. Select & rename relevant columns ──────────────────────────────
    .select(
        col("Id").cast("long").alias("id"),
        col("Score").cast("int").alias("score"),
        col("Time").cast("long").alias("time"),
        trim(col("Summary")).alias("summary"),
        trim(col("Text")).alias("review_text"),
    )

    # ── 2. Filter bad rows ────────────────────────────────────────────────
    .filter(col("review_text").isNotNull())
    .filter(length(col("review_text")) > 0)
    .filter(col("score").isNotNull())
    .filter(col("score").between(1, 5))

    # ── 3. Derive binary sentiment label ──────────────────────────────────
    # Score 4-5 -> Positive (1) | Score 1-3 -> Negative (0)
    .withColumn(
        "positive",
        when(col("score") >= 4, 1).otherwise(0).cast("int")
    )

    # ── 4. Add Bronze metadata columns ────────────────────────────────────
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", input_file_name())
    .withColumn("_record_id",   monotonically_increasing_id())
)

bronze_df.printSchema()
print(f"\nBronze row count: {bronze_df.count()}")
bronze_df.show(5, truncate=80)

In [ ]:
# ── Verify bronze output ────────────────────────────────────────────────────
print("Label distribution:")
bronze_df.groupBy("positive").count().orderBy("positive").show()

print("Score vs Positive label (sanity check):")
bronze_df.groupBy("score", "positive").count().orderBy("score").show()

print("Null check on bronze columns:")
bronze_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ["id", "score", "review_text", "positive", "_ingested_at", "_record_id"]
]).show()

## 4. Create a Validated 100-Row Sample

Select exactly 20 valid reviews from each score (1–5) and save one CSV object to S3.

In [ ]:
from pyspark.sql import Window
import boto3
import csv
import io

SAMPLE_BUCKET = "amazon-food-reviews-ml-model"
SAMPLE_KEY = "sample-dataset/Reviews_sample_100.csv"

# Reject shifted/malformed CSV records before sampling. Casting invalid values
# to a numeric type produces null, so those records do not pass these checks.
valid_reviews_df = raw_df.filter(
    F.col("Id").cast("long").isNotNull()
    & F.col("HelpfulnessNumerator").cast("int").isNotNull()
    & F.col("HelpfulnessDenominator").cast("int").isNotNull()
    & F.col("Score").cast("int").between(1, 5)
    & F.col("Time").cast("long").isNotNull()
    & F.col("Text").isNotNull()
    & (F.length(F.trim(F.col("Text"))) > 0)
    & (
        F.col("HelpfulnessNumerator").cast("int")
        <= F.col("HelpfulnessDenominator").cast("int")
    )
)

# Stop instead of silently producing an incomplete or corrupt sample.
score_counts = {
    int(row["Score"]): row["count"]
    for row in valid_reviews_df.groupBy("Score").count().collect()
}
missing_scores = {score: score_counts.get(score, 0) for score in range(1, 6) if score_counts.get(score, 0) < 20}
if missing_scores:
    raise ValueError(f"Not enough valid reviews for balanced sampling: {missing_scores}")

score_window = Window.partitionBy("Score").orderBy(F.rand(seed=42))
sample_100_df = (
    valid_reviews_df
    .withColumn("_sample_row", F.row_number().over(score_window))
    .filter(F.col("_sample_row") <= 20)
    .drop("_sample_row")
    .orderBy(F.rand(seed=42))
)

sample_rows = sample_100_df.collect()
if len(sample_rows) != 100:
    raise ValueError(f"Expected 100 sampled rows, got {len(sample_rows)}")

# Create one correctly quoted CSV file and upload it as one S3 object.
csv_buffer = io.StringIO(newline="")
writer = csv.DictWriter(
    csv_buffer,
    fieldnames=sample_100_df.columns,
    quoting=csv.QUOTE_MINIMAL,
    lineterminator="\n",
)
writer.writeheader()
writer.writerows(row.asDict(recursive=True) for row in sample_rows)

csv_bytes = csv_buffer.getvalue().encode("utf-8")
s3 = boto3.client("s3", region_name="eu-north-1")
s3.put_object(
    Bucket=SAMPLE_BUCKET,
    Key=SAMPLE_KEY,
    Body=csv_bytes,
    ContentType="text/csv",
)

# Verify both the sample balance and the uploaded S3 object.
sample_100_df.groupBy("Score").count().orderBy("Score").show()
uploaded = s3.head_object(Bucket=SAMPLE_BUCKET, Key=SAMPLE_KEY)
print(f"Saved {len(sample_rows)} valid rows ({uploaded['ContentLength']} bytes) to:")
print(f"s3://{SAMPLE_BUCKET}/{SAMPLE_KEY}")

In [ ]:
%stop_session